# Flappy Bird PPO — Inference-Only Evaluation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import sys

os.environ['SDL_VIDEODRIVER'] = 'dummy'
sys.path.insert(0, '../itml-project2')

from ple.games.flappybird import FlappyBird
from ple import PLE

In [ ]:
class PPO_Flappy(nn.Module):
    def __init__(self, num_layers, layer_specs):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(nn.Linear(layer_specs[i][0], layer_specs[i][1]))
        self.actor = nn.Linear(layer_specs[-1][1], 2)
        self.critic = nn.Linear(layer_specs[-1][1], 1)

    def forward(self, x):
        for i in self.layers:
            x = F.tanh(i(x))
        action_prob = F.softmax(self.actor(x), dim=-1)
        state_value = self.critic(x)
        return action_prob, state_value

In [ ]:
model = PPO_Flappy(5, [[8, 1024], [1024, 128], [128, 16], [16, 256], [256, 256]])
model.load_state_dict(torch.load('flappy weights/Parallel 6587.pt', weights_only=True))
model.eval()
print("Model loaded successfully.")

In [ ]:
def normalize_game_state(state):
    means = torch.tensor([256.0, 0.0, 200.0, 200.0, 200.0, 400.0, 200.0, 200.0])
    stds = torch.tensor([128.0, 5.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0])
    return (state - means) / stds

In [ ]:
NUM_EPISODES = 1

episode_rewards = []
episode_pipes = []
episode_lengths = []

game = FlappyBird()

with torch.no_grad():
    for ep in range(NUM_EPISODES):
        p = PLE(game, fps=30, display_screen=False, force_fps=True)
        p.init()
        p.reset_game()

        done = False
        total_reward = 0.0
        pipes_passed = 0
        steps = 0

        while not done:
            print_status = False
            state = normalize_game_state(torch.tensor(list(p.getGameState().values())))
            action_prob, _ = model(state)

            # Greedy action selection (deterministic)
            action = torch.argmax(action_prob).item()

            if action == 0:
                ret_action = p.getActionSet()[0]
            else:
                ret_action = p.getActionSet()[1]

            reward = p.act(ret_action)
            total_reward += reward
            steps += 1

            if reward > 0.9:
                pipes_passed += 1
                if pipes_passed %1000 == 0:
                    print_status = True

            if print_status:
                print(f"{pipes_passed} pipes passed")

            if p.game_over():
                done = True

        episode_rewards.append(total_reward)
        episode_pipes.append(pipes_passed)
        episode_lengths.append(steps)

        print(f"Episode {ep+1:3d}/{NUM_EPISODES} | "
              f"Reward: {total_reward:7.1f} | "
              f"Pipes: {pipes_passed:3d} | "
              f"Steps: {steps:5d}")

# Aggregate stats
rewards_arr = np.array(episode_rewards)
pipes_arr = np.array(episode_pipes)
lengths_arr = np.array(episode_lengths)

print("\n" + "="*55)
print(f"{'Aggregate Stats':^55}")
print("="*55)
print(f"  Mean reward:   {rewards_arr.mean():.1f}")
print(f"  Median reward: {np.median(rewards_arr):.1f}")
print(f"  Max reward:    {rewards_arr.max():.1f}")
print(f"  Min reward:    {rewards_arr.min():.1f}")
print(f"  Std reward:    {rewards_arr.std():.1f}")
print(f"  Mean pipes:    {pipes_arr.mean():.1f}")
print(f"  Mean steps:    {lengths_arr.mean():.0f}")
print("="*55)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(range(len(episode_rewards)), episode_rewards, color='steelblue', alpha=0.7, label='Episode reward')
ax.axhline(y=np.mean(episode_rewards), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean ({np.mean(episode_rewards):.1f})')

ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title(f'Evaluation \u2014 {NUM_EPISODES} Episodes | Best: {max(episode_rewards):.1f}')
ax.legend()
plt.tight_layout()
plt.show()